# 01 — First look at the data (EDA)

Goal: understand what is in each file, how much data we have, and what problems the raw sensors have.

**What each per-minute signal means physically (gas-lifted well):**

| Column | Meaning | What to expect |
|---|---|---|
| `whp_psia` | Wellhead pressure, upstream of the choke | Flowing: ~200–500 psia. Shut-in: jumps UP (choke closed, fluid stops, pressure from below is no longer released) |
| `whp_downstream_psia` | Pressure after the choke (flowline) | ~ separator/manifold pressure. Drops at shut-in (no flow coming through) |
| `dh_pressure_psia` | Downhole gauge pressure ≈ **Pwf** | Flowing: low (drawdown). Shut-in: **builds up** towards reservoir pressure → this curve is what PTA analyses |
| `dh_temperature_degf` | Downhole temperature | Flowing: hot fluid from reservoir. Shut-in: cools slowly toward the surrounding rock temperature at gauge depth |
| `annulus_pressure_psia` | Casing pressure = lift-gas pressure | Follows gas-lift injection |
| `gas_lift_rate_mscfd` | Lift gas injected | Drops to 0 when the well is shut in |

Pressures are **psia** (absolute): psia = psig + 14.7.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path('..', 'src').resolve()))
import load

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

hf = load.load_high_freq()          # ~3.9M rows; first call converts CSV -> parquet
tests = load.load_well_tests()
choke = load.load_choke_events()
pvt = load.load_pvt()
completion = load.load_completion()
shut_ins = choke[choke.choke_opening_pct == 0]       # choke closed = shut-in

SENSORS = ['whp_psia', 'whp_downstream_psia', 'dh_pressure_psia',
           'dh_temperature_degf', 'annulus_pressure_psia', 'gas_lift_rate_mscfd']
hf.head()

## 1. Coverage per well
If every minute were present, a well would have one row per minute from start to end. `missing_min` = gaps.

In [ ]:
g = hf.groupby('well_name')
overview = pd.DataFrame({'start': g.timestamp.min(), 'end': g.timestamp.max(), 'rows': g.size()})
overview['expected_min'] = ((overview.end - overview.start).dt.total_seconds() / 60 + 1).astype(int)
overview['missing_min'] = overview.expected_min - overview.rows
overview['tests'] = tests.well_name.value_counts()
overview

## 2. Data-quality problems in the raw sensors

1. **`-999.25` values** — this is the standard *null* code in oilfield data files (from the LAS log format).
   It means "no reading", NOT a real pressure. Must be turned into missing (NaN).
2. **Duplicate rows** — same well + same minute, recorded twice.
3. **Spikes** — e.g. downhole pressure jumping to 8000+ psia for one minute. Physically impossible
   (rock cannot change pressure that fast) → sensor glitch.
4. **Frozen sensor** — the same value repeated for hours (real sensors always have small noise).

In [ ]:
print('Null-code (-999.25) count per column:')
print((hf[SENSORS] < -900).sum())

print('\nExact duplicate rows:', hf.duplicated().sum())

# Frozen sensor: same value as previous minute, for >= 60 minutes in a row
print('\nMinutes inside flat runs >= 60 min:')
for c in SENSORS:
    same = (hf[c] == hf[c].shift()) & (hf.well_name == hf.well_name.shift())
    run_length = same.groupby((~same).cumsum()).transform('sum')
    print(f'  {c:24s} {int((run_length >= 60).sum())}')

hf[SENSORS].mask(hf[SENSORS] < -900).describe(percentiles=[.01, .5, .99]).T.round(1)

## 3. Plot one well over the whole period
Green bands = well tests (our labels). Red lines = shut-ins (choke closed, 0 %).
Null codes are hidden so we can see the real signal (spikes are still visible).

In [ ]:
def plot_well(well, start=None, end=None):
    d = hf[hf.well_name == well].set_index('timestamp')[SENSORS]
    d = d.mask(d < -900)                        # hide -999.25 null codes
    d = d.loc[start:end]
    t = tests[tests.well_name == well]
    s = shut_ins[shut_ins.well_name == well].rename(columns={'timestamp': 't_shut'})
    c = choke[choke.well_name == well]

    fig, axes = plt.subplots(len(SENSORS) + 1, 1, figsize=(15, 16), sharex=True)
    for ax, col in zip(axes, SENSORS):
        ax.plot(d.index, d[col], lw=0.4)
        ax.set_ylabel(col, fontsize=8)
        for _, row in t.iterrows():
            ax.axvspan(row.test_start, row.test_end, color='green', alpha=0.3)
        for ts in s.t_shut:
            ax.axvline(ts, color='red', ls='--', lw=1)
    axes[-1].step(c.timestamp, c.choke_opening_pct, where='post')
    axes[-1].set_ylabel('choke opening %')
    axes[-1].set_xlim(d.index.min(), d.index.max())
    fig.suptitle(f'{well}  (green = well tests, red = shut-ins)')
    fig.tight_layout()
    plt.show()

plot_well('GL-01')

## 4. Zoom on a shut-in (this is what PTA analyses)

On 15 Apr GL-01 choke goes to 0 % (closed) for 72 h:
- **Downhole pressure** jumps from ~3240 to ~3510 psia — fast at first, then slowly flattening.
  This *build-up* curve is the input for the Horner plot. How fast/how high it rises tells us
  permeability (k) and skin (s).
- **WHP** rises (choke closed, pressure trapped below it); **downstream pressure** drops.
- **Gas lift** goes to 0; **downhole temperature** cools from 203 to 198 °F.
- Re-opening is done in 3 steps (choke 21.9 → 29.7 → 37.5 %) — visible as steps in the pressure.

In [ ]:
plot_well('GL-01', '2025-04-14', '2025-04-20')

## 5. Shut-ins in our data
Every closure of a choke (opening = 0 %) is a shut-in: the downhole pressure builds up — the input for PTA.

In [ ]:
print('Closed-choke (0 %) events in choke file = shut-ins:')
shut_ins

In [ ]:
# Look at any other well, e.g.:
plot_well('GL-06')